# Fine-Tuning Qwen 3.5 0.8B with MLX and SarfTok

This notebook demonstrates how to fine-tune the highly efficient **Qwen 3.5 0.8B** model natively on Apple Silicon using `mlx_lm`.

More importantly, it shows how to use **SarfTok** *on top* of the Qwen tokenizer. By running SarfTok parallel to Qwen's byte-pair encoding (BPE), we extract high-confidence Arabic morphological features (like the exact root and grammatical pattern) and fuse them into the embedding layer. This "solves" the morphological blindness of BPE tokenizers without sacrificing Qwen's 4.60 tokens/word compression efficiency.

### Prerequisites
You need a Mac with Apple Silicon (M1/M2/M3/M4) to run `mlx` efficiently.

```bash
pip install mlx mlx-lm datasets camel-tools 
```

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import mlx.core as mx
import mlx.nn as nn
from datasets import load_dataset
from mlx_lm import load, generate
from mlx_lm.tuner import TrainingArgs, linear_to_lora_layers, train
from mlx_lm.tuner.utils import print_trainable_parameters


## 1. Load the Model and Base Tokenizer

We use `mlx_lm` to load the Qwen 3.5 0.8B model into unified memory.

In [3]:
model_id = "Qwen/Qwen3.5-0.8B"
model, qwen_tokenizer = load(model_id)

print("Model loaded successfully!")
print(f"Qwen Vocab Size: {qwen_tokenizer.vocab_size:,}")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model loaded successfully!
Qwen Vocab Size: 248,044


## 2. Load SarfTok (The Morphological Injector)

While Qwen handles the surface tokens, SarfTok analyzes the Arabic text and aligns the linguistic metadata (Roots, Patterns, Clitics) to Qwen's BPE tokens.

We initialize the `CamelMorphAnalyzer` which uses the CAMeL Tools backend to get highly accurate roots for derived Arabic verbs and nouns.

In [4]:
from sarftok.morph_analyzer.camel_wrapper import CamelMorphAnalyzer
from sarftok.segmenter import ArabicSegmenter

# Initialize the morphological backend
sarftok_analyzer = CamelMorphAnalyzer(db_name='calima-msa-r13', top_k=1)
sarftok_segmenter = ArabicSegmenter()

print("SarfTok Morphological backend loaded.")

SarfTok Morphological backend loaded.


## 3. The SarfTok Tokenizer Wrapper

To train with `mlx_lm`, we need raw text datasets. However, `mlx_lm` handles tokenization internally during training. 

To inject SarfTok into MLX, we create a dataset wrapper that pre-computes the Qwen tokens *and* the SarfTok morphological embeddings, saving them as MLX arrays.

In [ ]:
class SarfTokMLXDataset:
    """
    Align Qwen BPE tokens with SarfTok morphological features for MLX tuning.
    Outputs (tokens, offset) pairs expected by mlx_lm.tuner.train.
    """
    def __init__(self, data_list, qwen_tok, morph_analyzer, segmenter, max_length=1024):
        self.records = []
        self.max_length = max_length
        eos_id = getattr(qwen_tok, 'eos_token_id', None)

        for text in data_list:
            token_ids = qwen_tok.encode(text)
            if eos_id is not None and (len(token_ids) == 0 or token_ids[-1] != eos_id):
                token_ids.append(eos_id)

            words = segmenter.segment_flat(text)
            analyses = morph_analyzer.analyze_sentence(words, context=text)

            morph_snapshot = []
            for word, word_analyses in zip(words, analyses):
                morph_snapshot.append({
                    "word": word,
                    "analyses": [
                        {
                            "root": analysis.root,
                            "pattern": analysis.pattern,
                            "prob": float(analysis.prob),
                            "pos": analysis.pos,
                        }
                        for analysis in word_analyses
                    ],
                })

            for chunk in self._chunk_sequence(token_ids, eos_id):
                if len(chunk) < 2:
                    continue
                self.records.append({
                    "tokens": chunk,
                    "offset": 0,
                    "morph": morph_snapshot,
                })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        return rec["tokens"], rec["offset"]

    def morph_view(self, idx):
        """Return cached morphology for inspection/debugging."""
        return self.records[idx]["morph"]

    def _chunk_sequence(self, tokens, eos_id):
        if not self.max_length or self.max_length <= 0:
            yield tokens
            return
        step = self.max_length
        for start in range(0, len(tokens), step):
            chunk = tokens[start:start + step]
            if not chunk:
                continue
            if eos_id is not None and chunk[-1] != eos_id:
                if len(chunk) >= self.max_length:
                    chunk = chunk[:-1] + [eos_id]
                else:
                    chunk = chunk + [eos_id]
            yield chunk


## 4. Setting up LoRA Adapters

To keep Apple Silicon memory usage in check, we only adapt the final decoder blocks with low-rank LoRA (rank=8, dropout=0). If you have more headroom, increase `num_lora_layers` or the rank; otherwise shrink them further.


In [6]:
model.freeze()  # Freeze the base model parameters

# Configure memory-friendly LoRA layers
num_lora_layers = min(8, len(model.layers))  # adapt only the last 8 decoder blocks
lora_config = {
    "rank": 4,
    "scale": 8.0,
    "dropout": 0.0,
}

linear_to_lora_layers(model, num_layers=num_lora_layers, config=lora_config)
print_trainable_parameters(model)


Trainable parameters: 0.120% (0.902M/752.392M)


## 5. Prepare the Arabic Training Dataset

We take 1,000 multi-turn chat samples from the [FreedomIntelligence/sharegpt-arabic](https://huggingface.co/datasets/FreedomIntelligence/sharegpt-arabic) dataset, which contains ShareGPT-style Arabic conversations covering instructions, reasoning, and open-ended QA.


In [7]:
dataset_id = "FreedomIntelligence/sharegpt-arabic"
raw_dataset = load_dataset(dataset_id, split="train[:1000]")

def format_turn(role: str, text: str) -> str:
    role = (role or "").lower()
    label = "### السؤال" if role in {"user", "human"} else "### الإجابة"
    return f"{label}\n{text.strip()}"

def format_example(example):
    conversations = example.get("conversations") or example.get("messages") or []
    formatted = []
    for turn in conversations:
        content = (turn.get("value") or turn.get("content") or "").strip()
        if not content:
            continue
        formatted.append(format_turn(turn.get("from") or turn.get("role") or "", content))
    return "".join(formatted)

train_data_raw = [format_example(rec) for rec in raw_dataset]
print(f"Loaded {len(train_data_raw)} ShareGPT-style samples from {dataset_id}.")


Loaded 1000 ShareGPT-style samples from FreedomIntelligence/sharegpt-arabic.


## 6. Defining the Custom Forward Pass (Morphological Fusion)

If we were directly modifying the MLX Transformer model architecture to use SarfTok embeddings, we would intercept the `model.embed_tokens` layer to add the SarfTok representations.

```python
# Conceptual Architecture Modification for SarfTok Fusion
class MorphoSarfTokEmbedding(nn.Module):
    def __init__(self, vocab_size, morph_vocab_size, hidden_dim):
        super().__init__()
        self.surface_emb = nn.Embedding(vocab_size, hidden_dim)
        self.morph_emb = nn.Embedding(morph_vocab_size, hidden_dim // 4) 
        self.proj = nn.Linear(hidden_dim + (hidden_dim // 4), hidden_dim)
        
    def __call__(self, input_ids, root_ids):
        surface_x = self.surface_emb(input_ids)
        morph_x = self.morph_emb(root_ids)
        
        # Concatenate explicit morphology with base representation
        fused = mx.concatenate([surface_x, morph_x], axis=-1)
        return self.proj(fused)
```

Since modifying the standard Qwen architecture requires a custom MLX model definition, for this standard LoRA tuning script, we simply train the adapters using the base `mlx_lm` trainer.

## 7. Start Training

Using the MLX-LM trainer to fine-tune the LoRA adapters on Apple Silicon.

In [ ]:
from pathlib import Path
import mlx.optimizers as optim

adapter_dir = Path("adapters")
adapter_dir.mkdir(exist_ok=True)

# 1. Define Optimizer
optimizer = optim.Adam(learning_rate=2e-5)

# 2. Training Loop Arguments
training_args = TrainingArgs(
    batch_size=1,
    iters=100,
    steps_per_report=10,
    steps_per_eval=20,
    steps_per_save=50,
    max_seq_length=1024,
    adapter_file=str(adapter_dir / "adapters.safetensors"),
)

print("Training configuration ready. Call train() to begin fine-tuning!")

# 3. Run the trainer
# train(
#     model=model,
#     optimizer=optimizer,
#     train_dataset=train_dataset,
#     val_dataset=train_dataset,
#     args=training_args,
# )


## 7c. Full Training Run with Metrics

When you are ready for the full adapter job, run the cell below to launch MLX training on the entire ShareGPT subset. It prints per-report metrics and refreshes the adapters on disk.


In [ ]:
print("
[ Full Run ] Building SarfTok dataset for all samples...")
train_dataset = SarfTokMLXDataset(train_data_raw, qwen_tokenizer, sarftok_analyzer, sarftok_segmenter, max_length=1024)
dataset_size = len(train_dataset)
full_epochs = 3
full_iters = dataset_size * full_epochs
print(f"[ Full Run ] Dataset ready: {dataset_size} examples (chunked).")
print(f"[ Full Run ] Training for {full_epochs} epochs (~{full_iters} steps)...")

full_optimizer = optim.Adam(learning_rate=2e-5)
full_args = TrainingArgs(
    batch_size=1,
    iters=full_iters,
    steps_per_report=50,
    steps_per_eval=200,
    val_batches=5,
    max_seq_length=1024,
    steps_per_save=500,
    adapter_file=str(adapter_dir / "adapters.safetensors"),
)

print("[ Full Run ] Starting training...")
train(
    model=model,
    optimizer=full_optimizer,
    train_dataset=train_dataset,
    val_dataset=train_dataset,
    args=full_args,
)
print("[ Full Run ] Training complete. Adapters saved to", adapter_dir)


## 7b. Quick Smoke Test (Optional)

Before running the full adapter job on all 1,000 ShareGPT-style samples, we spin up a miniature dataset (32 chats) to verify that SarfTok preprocessing, LoRA injection, and MLX training all run end-to-end.


In [ ]:
# Build a tiny SarfTok dataset from the first 32 ShareGPT conversations
smoke_subset = train_data_raw[:32]
smoke_dataset = SarfTokMLXDataset(smoke_subset, qwen_tokenizer, sarftok_analyzer, sarftok_segmenter, max_length=1024)
print(f"Smoke-test subset size: {len(smoke_dataset)} examples.")

# Fresh optimizer/args bundle so the smoke test is isolated from the real run
smoke_optimizer = optim.Adam(learning_rate=2e-5)
smoke_args = TrainingArgs(
    batch_size=1,
    iters=2,
    steps_per_report=1,
    steps_per_eval=2,
    val_batches=1,
    max_seq_length=1024,
    adapter_file=str(adapter_dir / "adapters-smoke.safetensors"),
)

print("
[ Smoke Test ] Starting training run...")
train(
    model=model,
    optimizer=smoke_optimizer,
    train_dataset=smoke_dataset,
    val_dataset=smoke_dataset,
    args=smoke_args,
)
print("[ Smoke Test ] Completed.")


## 8. Generation with Fusing

Once the adapters are saved (for example in `adapters/adapters.safetensors`), you can load them and generate text. The Qwen tokenizer handles surface tokens while the LoRA adapters inject the learned morphology-aware corrections.


In [14]:
adapter_weights = adapter_dir / "adapters.safetensors"
adapter_config = adapter_dir / "adapter_config.json"
use_adapters = adapter_weights.exists() and adapter_config.exists()
print(f"Loading model for generation (using adapters: {use_adapters})")

if use_adapters:
    infer_model, infer_tokenizer = load(model_id, adapter_path=str(adapter_dir))
else:
    infer_model, infer_tokenizer = load(model_id)

default_prompt = """<|im_start|>user
لخص فوائد التعلم العميق في الرعاية الصحية
<|im_end|>
<|im_start|>assistant
"""
prompt = default_prompt.strip()
response = generate(
    infer_model,
    infer_tokenizer,
    prompt=prompt,
    max_tokens=400,
    verbose=True,
)

print("""
--- Generated Response ---
""")
print(response)


Loading model for generation (using adapters: False)


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


<think>

</think>

التعلم العميق (Deep Learning) أحدث ثورة في مجال الرعاية الصحية، حيث تم تطبيقه على نطاق واسع لتحليل البيانات الضخمة، مما أدى إلى تحسينات كبيرة في التشخيص، التنبؤ بالأمراض، وإدارة الموارد. إليك أبرز الفوائد التي تميل إليها:

### 1. التشخيص الدقيق والتنبؤ بالأمراض
من خلال تحليل البيانات الكبيرة (Big Data)، يمكن للذكاء الاصطناعي اكتشاف الأنماط التي قد لا يراها البشر.
*   **التشخيص المبكر:** يساعد في تحديد الحالات الخطيرة (مثل السرطان أو الأمراض المزمنة) قبل ظهور الأعراض، مما يحد من المضاعفات.
*   **التنبؤ بالأمراض:** يمكن أن يوقع الأطباء على احتمالية حدوث مرض معين في المستقبل بناءً على التاريخ الطبي والبيانات الجينية.

### 2. تحسين دقة التشخيص الطبي
*   **الذكاء الاصطناعي:** يمكنه تحليل الصور الطبية (مثل الأشعة السينية، الرنين المغناطيسي، والتصوير المقطعي) بدقة غير مسبوقة، مما يقلل من الأخطاء البشرية.
*   **التعرف على الأنماط:** يكتشف العلاقات المعقدة بين الأعراض والعلامات المرضية التي يصعب على الأطباء استنتاجها يدوياً.

### 3. إدارة الموارد الصحية بكفاءة
*   **التنبؤ ب